# Decision Tree



---

## What is a Decision Tree?

A **Decision Tree** is a supervised learning algorithm used for **classification** and **regression**.
It works like a flowchart:

* **Internal nodes** → ask a question about a feature
* **Branches** → outcomes of that question
* **Leaf nodes** → final prediction (class label or numeric value)

Think: *“If this condition is true, go left; otherwise, go right.”*

---

## Example (Classification)

Imagine predicting **whether someone will buy a product**:

```
Is age > 30?
├── Yes → Is income high?
│   ├── Yes → Buy
│   └── No → Not Buy
└── No → Buy
```

Each split is a decision based on data.

---

## Types of Decision Trees

### 1. Classification Trees

* Output: **class labels**
* Examples: spam vs not spam, disease vs no disease
* Metrics used:

  * Gini Impurity
  * Entropy / Information Gain

### 2. Regression Trees

* Output: **continuous values**
* Examples: house price, temperature
* Metrics used:

  * Mean Squared Error (MSE)
  * Mean Absolute Error (MAE)

---

## How a Decision Tree Learns

The goal is to split the data in a way that makes the resulting groups as **pure** as possible.

### Step-by-step:

1. Start with the full dataset
2. Try all possible splits on all features
3. Choose the split that best separates the data
4. Repeat recursively for each child node
5. Stop when a stopping condition is met

---

## Splitting Criteria (Very Important)

### 1. Gini Impurity

Measures how often a randomly chosen element would be misclassified.

[
Gini = 1 - \sum p_i^2
]

* Lower = purer node
* Used by **CART**

---

### 2. Entropy & Information Gain

Entropy measures disorder:

[
Entropy = - \sum p_i \log_2 p_i
]

Information Gain:

[
IG = Entropy(parent) - \sum weighted\ Entropy(children)
]

* Higher IG = better split
* Used by **ID3 / C4.5**

---

### 3. Variance Reduction (Regression)

Choose the split that minimizes variance in child nodes.

---

## Stopping Criteria

A tree stops growing when:

* Maximum depth is reached
* Node has too few samples
* All samples belong to the same class
* No split improves performance

---

## Overfitting Problem 🚨

Decision Trees **love to overfit**.

* Deep tree → memorizes data
* Performs great on training data, poorly on test data

### Solutions:

#### 1. Pre-Pruning

* Limit max depth
* Minimum samples per leaf
* Minimum samples to split

#### 2. Post-Pruning

* Grow full tree
* Remove branches that don’t improve validation performance

---

## Advantages

✅ Easy to understand and interpret
✅ No feature scaling needed
✅ Handles numerical & categorical data
✅ Non-linear relationships supported

---

## Disadvantages

❌ Overfitting
❌ Unstable (small data change → big tree change)
❌ Greedy algorithm (locally optimal splits)
❌ Bias toward features with more levels

---

## Popular Decision Tree Algorithms

| Algorithm | Split Metric     | Notes                  |
| --------- | ---------------- | ---------------------- |
| ID3       | Information Gain | Categorical only       |
| C4.5      | Gain Ratio       | Handles missing values |
| CART      | Gini / MSE       | Binary splits only     |
| CHAID     | Chi-square       | Statistical focus      |

---

## Decision Trees vs Other Models

* vs **Linear models** → trees handle non-linearity better
* vs **KNN** → faster inference
* vs **Neural Networks** → more interpretable, less powerful

---

## Decision Trees in Practice

Used in:

* Credit scoring
* Medical diagnosis
* Customer segmentation
* Fraud detection

And they’re the **building blocks** of:

* Random Forest 🌲🌲🌲
* Gradient Boosting
* XGBoost, LightGBM, CatBoost

---

## TL;DR

* Decision Trees split data using feature-based questions
* Simple, interpretable, and powerful
* Prone to overfitting → use pruning or ensembles
* Foundation of many state-of-the-art ML models



---

# 1. Impurity Measures (Heart of Decision Trees)

A decision tree grows by **reducing impurity** at every split.

## What is “impurity”?

Impurity = *how mixed the classes are in a node*

* Pure node → all samples belong to one class
* Impure node → mixture of classes

---

## 1.1 Gini Impurity

### Formula

[
\text{Gini} = 1 - \sum_{i=1}^{C} p_i^2
]

where

* (p_i) = proportion of class (i) in the node
* (C) = number of classes

---

### Intuition

* Measures **probability of misclassification**
* If you randomly label a point according to class distribution, how often would you be wrong?

---

### Properties

| Case            | Gini |
| --------------- | ---- |
| Perfectly pure  | 0    |
| Binary (50–50)  | 0.5  |
| Multi-class max | < 1  |

---

### Example

Node with 10 samples:

* 6 Positive, 4 Negative

[
Gini = 1 - (0.6^2 + 0.4^2)
= 1 - (0.36 + 0.16)
= 0.48
]

---

### Why CART prefers Gini

* Faster to compute (no logs)
* Works well empirically
* Similar behavior to entropy

---

## 1.2 Entropy

### Formula

[
\text{Entropy} = - \sum_{i=1}^{C} p_i \log_2(p_i)
]

---

### Intuition

* Borrowed from **information theory**
* Measures **uncertainty / randomness**
* Answers: *how much information is needed to classify a sample?*

---

### Properties

| Case            | Entropy    |
| --------------- | ---------- |
| Pure node       | 0          |
| Binary (50–50)  | 1          |
| Multi-class max | (\log_2 C) |

---

### Example

Same node (6+, 4−):

[
Entropy = - (0.6\log_2 0.6 + 0.4\log_2 0.4)
≈ 0.97
]

---

### Gini vs Entropy (Deep Comparison)

| Aspect       | Gini    | Entropy              |
| ------------ | ------- | -------------------- |
| Computation  | Faster  | Slower (log)         |
| Sensitivity  | Less    | More to rare classes |
| Output scale | [0, <1] | [0, log C]           |
| Used by      | CART    | ID3, C4.5            |

👉 In practice: **almost identical splits**

---

## 1.3 Information Gain (IG)

Entropy alone is not enough — we care about **how much it decreases after a split**.

### Formula

[
IG(S, A) = Entropy(S) - \sum_{v \in values(A)} \frac{|S_v|}{|S|} Entropy(S_v)
]

Where:

* (S) = parent dataset
* (A) = feature
* (S_v) = subset after split

---

### Intuition

“How much uncertainty did this feature remove?”

---

### Example Logic (no numbers)

* Parent entropy = high
* Child entropies = low
* Weighted average = low
  → **High Information Gain**

---

### Bias Problem of IG 🚨

* Prefers features with **many unique values**
* Example: `Customer_ID` (unique for each row)

  * Creates pure leaves
  * Zero generalization

---

## 1.4 Gain Ratio (Fix for IG Bias)

Used in **C4.5**

### Formula

[
Gain\ Ratio = \frac{Information\ Gain}{Split\ Information}
]

Where:
[
Split\ Info = - \sum \frac{|S_v|}{|S|} \log_2 \frac{|S_v|}{|S|}
]

---

### Intuition

* Penalizes features that create many small splits
* Normalizes Information Gain

---

## 2. Regression Tree Criterion

For regression, classes don’t exist → use **variance**

### Variance

[
Var = \frac{1}{n}\sum (y_i - \bar{y})^2
]

### Goal

Minimize:
[
\text{Weighted Variance of Children}
]

Equivalent to minimizing **MSE**

---

# 3. Pruning (Controlling Overfitting)

Without pruning → tree grows until **memorization**

---

## 3.1 Pre-Pruning (Early Stopping)

Stop tree growth **while building**

### Common Techniques

| Method                | Meaning                     |
| --------------------- | --------------------------- |
| max_depth             | Limit tree height           |
| min_samples_split     | Min samples to split a node |
| min_samples_leaf      | Min samples in leaf         |
| min_impurity_decrease | Minimum gain required       |

---

### Pros

✅ Fast
✅ Prevents extreme overfitting

### Cons

❌ Might stop too early
❌ Misses important patterns

---

## 3.2 Post-Pruning (Cost-Complexity Pruning)

Grow full tree → **cut back**

### Idea

Balance:
[
\text{Error} + \alpha \times \text{Tree Size}
]

Where:

* α controls penalty for complexity

---

### Cost-Complexity Function

[
R_\alpha(T) = R(T) + \alpha |T|
]

* (R(T)) = misclassification error
* (|T|) = number of leaves

---

### Process

1. Grow full tree
2. Iteratively remove weakest branches
3. Validate performance
4. Choose optimal α

Used in **CART**

---

### Pros

✅ Better generalization
✅ Theoretically sound

### Cons

❌ Slower
❌ More computation

---

## 4. Bias–Variance Tradeoff (Very Exam-Friendly)

| Tree Type    | Bias | Variance |
| ------------ | ---- | -------- |
| Shallow tree | High | Low      |
| Deep tree    | Low  | High     |

Pruning moves the tree **toward balance**

---

## 5. Why Trees Are Greedy

* Chooses best split *now*
* No backtracking
* No global optimization

Result:

* Not guaranteed optimal tree
* But fast and scalable

---

## 6. Interview Gold Nuggets ✨

* Gini ≈ Entropy → same splits mostly
* IG is biased toward high-cardinality features
* Gain Ratio fixes IG bias
* Pre-pruning = early stop
* Post-pruning = grow then cut
* CART uses **binary splits only**
* Trees are **high variance models**

---



## Implementations (via GPT): {Must redo!!}

In [ ]:
import numpy as np
from collections import Counter

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    # Gini impurity
    def gini(self, y):
        counts = Counter(y)
        impurity = 1.0
        total = len(y)
        for label in counts:
            p = counts[label] / total
            impurity -= p ** 2
        return impurity

    # Split dataset
    def split(self, X, y, feature, threshold):
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold
        return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

    # Best split
    def best_split(self, X, y):
        best_gain = 0
        best_feature = None
        best_threshold = None

        parent_gini = self.gini(y)
        n_features = X.shape[1]

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

                if len(y_l) == 0 or len(y_r) == 0:
                    continue

                weighted_gini = (
                    len(y_l) / len(y) * self.gini(y_l)
                    + len(y_r) / len(y) * self.gini(y_r)
                )

                gain = parent_gini - weighted_gini

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    # Build tree
    def build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        num_labels = len(set(y))

        # Stopping conditions
        if (
            num_labels == 1
            or num_samples < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return Counter(y).most_common(1)[0][0]

        feature, threshold = self.best_split(X, y)
        if feature is None:
            return Counter(y).most_common(1)[0][0]

        X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(X_l, y_l, depth + 1),
            "right": self.build_tree(X_r, y_r, depth + 1),
        }

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    # Predict one sample
    def _predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        if x[tree["feature"]] <= tree["threshold"]:
            return self._predict_one(x, tree["left"])
        else:
            return self._predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
X = np.array([
    [2, 3],
    [1, 5],
    [3, 2],
    [8, 7],
    [9, 6]
])

y = np.array([0, 0, 0, 1, 1])

dt = DecisionTree(max_depth=3)
dt.fit(X, y)

preds = dt.predict(X)
print(preds)


In [ ]:
import numpy as np

class DecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    # Mean Squared Error
    def mse(self, y):
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)

    def split(self, X, y, feature, threshold):
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold
        return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

    def best_split(self, X, y):
        best_gain = 0
        best_feature = None
        best_threshold = None

        parent_mse = self.mse(y)
        n_features = X.shape[1]

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

                if len(y_l) == 0 or len(y_r) == 0:
                    continue

                weighted_mse = (
                    len(y_l) / len(y) * self.mse(y_l)
                    + len(y_r) / len(y) * self.mse(y_r)
                )

                gain = parent_mse - weighted_mse

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build_tree(self, X, y, depth=0):
        if (
            len(y) < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return np.mean(y)

        feature, threshold = self.best_split(X, y)
        if feature is None:
            return np.mean(y)

        X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(X_l, y_l, depth + 1),
            "right": self.build_tree(X_r, y_r, depth + 1),
        }

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def _predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        if x[tree["feature"]] <= tree["threshold"]:
            return self._predict_one(x, tree["left"])
        else:
            return self._predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5]
])

y = np.array([1.2, 1.9, 3.1, 3.8, 5.2])

dt = DecisionTreeRegressor(max_depth=3)
dt.fit(X, y)

preds = dt.predict(X)
print(preds)
